[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anita-agasaveeran/Assignment4/blob/main/02_SEMMA_Flight_Delays/notebooks/SEMMA_Flight_Disruption.ipynb)

# Will My Flight Be Disrupted Tomorrow? A SEMMA Study of 11.7 Million US Flights
**CMPE 255 · Assignment 4 · Project 2 of 3**

This project follows SAS's **SEMMA** methodology: **S**ample → **E**xplore → **M**odify → **M**odel → **A**ssess, plus a scoring/deployment step. SEMMA has no business-understanding stage of its own, so a short **front pass** comes first. It frames the problem, checks the data, and freezes the train/validation/test partition before any modeling.

*Data: [US Flight Delays & Cancellations 2025-2026](https://www.kaggle.com/datasets/samuelpombourcq/us-flight-delays-and-cancellations-2025-2026) (Samuel_Pbq, CC0), from the US Bureau of Transportation Statistics (BTS) On-Time Performance table. 11,730,351 flights, Jan 2025 – Aug 2026.*

## 0. Setup

In [1]:
import os, subprocess, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/anita-agasaveeran/Assignment4.git"
if IN_COLAB:
    if not Path("/content/Assignment4").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Assignment4"], check=True)
    PROJECT_DIR = Path("/content/Assignment4/02_SEMMA_Flight_Delays")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/Assignment4/requirements.txt"], check=True)
    # The 314 MB flight file comes from Kaggle: add your kaggle.json (or KAGGLE_USERNAME / KAGGLE_KEY) first.
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src")); os.chdir(PROJECT_DIR)
print("Project dir:", PROJECT_DIR)

Project dir: /Users/anitaagasaveeran/Desktop/MSSE_Fall2026/CMPE255/Assignment4/02_SEMMA_Flight_Delays


In [2]:
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn, lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from flightdelay import config
from flightdelay.data import load_flights, period, assert_no_test_targets
from flightdelay.features import schedule_features, previous_day_rates
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | sklearn {sklearn.__version__} | lightgbm {lgb.__version__}")

python 3.12.14 | pandas 3.1.0 | sklearn 1.9.1 | lightgbm 4.7.0


---
# Front pass · Problem framing, data checks, frozen partition

> **Revision 2**, after critic round 1 (6/10; `critiques/frontpass_round1.md`). Main changes:
> - a **target leak** is removed (the tail-number feature, whose missingness means "cancelled");
> - previous-day rates now use a **17:00 decision clock**;
> - **two baselines** (schedule rule and day-before rule) instead of one;
> - the primary metric is **per-day** top-k lift, with day-clustered intervals;
> - a **summer check** runs inside the non-test data;
> - a seal guard is added, and the Hawaiian → Alaska recoding is handled.

> SEMMA starts at *Sample*. The course slides note that, unlike CRISP-DM, it has no business-understanding step. This front pass supplies the minimum: who decides what, with which information, and how success is judged. It also checks the data before any modeling.

### F.1 Problem
**Hypothetical stakeholder:** the customer-care team of a travel-booking service (or an airline). At **17:00 local the day before**, it can proactively handle passengers on about **2% of tomorrow's flights** (~380 of ~19k a day): rebooking tight connections, sending alerts, staffing help desks.

**Decision:** which of tomorrow's flights fall in that top 2% by risk.

**Allowed inputs:** the published schedule, plus outcomes of flights **scheduled to land by 17:00** on previous days. Nothing observed later.

**Target, `disrupted`:** arrived 15+ minutes late, cancelled, or diverted. That's 23.7% of flights in train + validation; the test months stay sealed.

**Scored population (assumption A-pop):** flights *with* a tail number. Every flight without one is cancelled (22,511 of 22,511 in train + validation). We *assume* these are early cancellations the team already knows about, but nothing in the data shows they were cancelled before 17:00 the day before, and they cluster on storm days. So Assess reports results **both ways**: with them excluded, and with them scored as disrupted.

**Success (Assess stage, sealed test months):**
- **Primary:** per-day top-2% precision and lift (precision ÷ that day's disruption rate), averaged over days. Top-10% is also reported.
- **Secondary:** pooled ROC-AUC / PR-AUC and calibration.
- **Always compared with two simple rules:**
  - schedule rule: airline × departure-hour history;
  - **day-before rule:** the mean of yesterday's origin, destination and carrier disruption rates.
- Exact targets are set in the Sample stage, from validation only.

### F.2 Data checks (all 11.7M flights; structure only, no target statistics on the test months)

In [3]:
cols = ["flight_date", "day_of_week", "airline_code", "tail_number", "flight_number", "origin", "dest",
        "sched_dep_time", "sched_arr_time", "dep_delay", "dep_del15", "arr_delay", "arr_del15", "cancelled", "diverted",
        "sched_elapsed_time", "distance", "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]
d = load_flights(columns=cols)
d["period"] = period(d["flight_date"])
operated = (d["cancelled"] == 0) & (d["diverted"] == 0)
late = operated & (d["arr_delay"] >= 15)
causes = d.loc[late, ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]].sum(axis=1)
dd = d[d["dep_delay"].notna()]
dup = d[d.duplicated(["flight_date", "airline_code", "flight_number", "origin", "sched_dep_time"], keep=False)]
per_day = d.groupby("flight_date").size()
checks = pd.Series({
    "flights": len(d),
    "date range": f"{d.flight_date.min().date()} → {d.flight_date.max().date()}",
    "flights per day (min / median / max)": f"{per_day.min():,} / {int(per_day.median()):,} / {per_day.max():,}",
    "carriers / airports": f"{d.airline_code.nunique()} / {pd.concat([d.origin.astype(str), d.dest.astype(str)]).nunique()}",
    "arr_del15 == (arr_delay ≥ 15), operated": f"{(d.loc[operated, 'arr_del15'] == (d.loc[operated, 'arr_delay'] >= 15)).mean():.4%}",
    "dep_del15 == (dep_delay ≥ 15)": f"{(dd.dep_del15 == (dd.dep_delay >= 15)).mean():.4%}",
    "delay causes sum to arr_delay (late flights)": f"{(causes == d.loc[late, 'arr_delay']).mean():.4%}",
    "cancelled flights that still have a dep_delay (left gate, then cancelled)": int((d.cancelled.eq(1) & d.dep_delay.notna()).sum()),
    "sched_elapsed_time missing / ≤ 0": f"{int(d.sched_elapsed_time.isna().sum())} / {int((d.sched_elapsed_time <= 0).sum())}",
    "sched_dep_time == 2400 (midnight)": int((d.sched_dep_time == 2400).sum()),
    "duplicate flight keys (rows)": f"{len(dup)} rows: " + ", ".join(f"{r.airline_code} {r.flight_number} {r.flight_date.date()}" for r in dup.head(2).itertuples()),
    "missing tail_number (train + valid): all cancelled?": f"{d.loc[d.tail_number.isna() & d.period.isin(['train', 'valid'])].shape[0]:,} rows, "
        f"{d.loc[d.tail_number.isna() & d.period.isin(['train', 'valid']), 'cancelled'].mean():.1%} cancelled",
})
display(checks.to_frame("value"))

,value
flights,11730351
date range,2025-01-01 → 2026-08-31
flights per day (min / median / max),"11,001 / 19,677 / 21,576"
carriers / airports,14 / 361
"arr_del15 == (arr_delay ≥ 15), operated",100.0000%
dep_del15 == (dep_delay ≥ 15),100.0000%
delay causes sum to arr_delay (late flights),99.9944%
"cancelled flights that still have a dep_delay (left gate, then cancelled)",8243
sched_elapsed_time missing / ≤ 0,4 / 14
sched_dep_time == 2400 (midnight),1


In [4]:
# Carrier mix by period (counts only): HA stops after 2025; its network is reported as AS from 2026; NK winds down
mix = d.groupby(["period", "airline_code"], observed=True).size().unstack("period")[["train", "valid", "test"]]
hawaii = d[d.airline_code.eq("AS") & (d.origin.astype(str).isin(["HNL", "OGG", "KOA", "LIH", "ITO"]) | d.dest.astype(str).isin(["HNL", "OGG", "KOA", "LIH", "ITO"]))]
display(mix.loc[["AS", "HA", "NK"]].assign(note=["HA network added from 2026-01", "last flight 2025-12-31", "last flight 2026-05-01"]))
print("AS flights touching Hawaii per month: 2025 mean", int(hawaii[hawaii.flight_date.dt.year == 2025].groupby(hawaii.flight_date.dt.to_period("M")).size().mean()),
      "| 2026 mean", int(hawaii[hawaii.flight_date.dt.year == 2026].groupby(hawaii.flight_date.dt.to_period("M")).size().mean()))

period,train,valid,test,note
airline_code,,,,
AS,245588.0,105464.0,122142.0,HA network added from 2026-01
HA,80084.0,NaN,NaN,last flight 2025-12-31
NK,194515.0,46848.0,354.0,last flight 2026-05-01


AS flights touching Hawaii per month: 2025 mean 1384 | 2026 mean 8063


**What this shows:**
- **The table is internally consistent.** Delay flags match delay minutes; delay causes add up on 99.994% of late flights.
- **Small, explained exceptions:**
  - 8,243 cancelled flights still have a `dep_delay`: they left the gate before being cancelled, as documented.
  - 4 flights are missing `sched_elapsed_time` and 14 have a value ≤ 0. Decision: drop them in Modify.
  - 1 flight is scheduled at 2400 (midnight); it gets departure hour 23.
  - The 1 duplicate key is two F9 rows with null delays; one is kept.
- **Days look complete,** including August 2026. BTS can revise its latest month, which is noted as a limitation.
- **Carrier changes are real and handled:**
  - Hawaiian's network is reported under Alaska from 2026, so `carrier` maps HA → AS (`config.CARRIER_GROUP`).
  - Spirit winds down, with its last flights on 2026-05-01 (the test period's first day).
- **Missing tail numbers = cancellations.** Every flight without a tail number is cancelled, so the v1 feature `leg_of_day` (missing for exactly these flights) **leaked the target**. Those flights are now excluded from the scored population, and no feature is derived from `tail_number`.

### F.3 Frozen partition and seal

In [5]:
part = d.groupby("period").agg(flights=("flight_date", "size"), first=("flight_date", "min"), last=("flight_date", "max"))
dev = assert_no_test_targets(d[d["period"].isin(["train", "valid"])])     # guard: no target stats on test
part["disrupted rate"] = dev.groupby("period")["disrupted"].mean()
display(part.loc[["train", "valid", "test"]])
print(f"train + valid disruption rate: {dev['disrupted'].mean():.3f}")

,flights,first,last,disrupted rate
period,,,,
train,7001619,2025-01-01,2025-12-31,0.236627
valid,2269061,2026-01-01,2026-04-30,0.236289
test,2459671,2026-05-01,2026-08-31,NaN


train + valid disruption rate: 0.237


| Partition | Months | Use |
|---|---|---|
| Train | Jan–Dec 2025 | Fit |
| Validation | Jan–Apr 2026 | Tuning, model choice, targets |
| *Summer check* | *train Jan–Apr 2025 → validate May–Aug 2025* | *Season-matched check, inside the non-test data, because the test is a summer* |
| **Test (sealed)** | **May–Aug 2026** | **Scored once in Assess**, after refitting on train + validation (refit policy) |

*Disclosures:*
- The v1 data check printed quarterly disruption rates for the whole period (test months included).
- The v1 report quoted an all-period rate (24.4%). That's now replaced by the train + validation rate.
- Neither informs any modeling choice. `assert_no_test_targets` now guards the target statistics in F.3 and F.5.

### F.4 Leakage audit
| Column / file | Known at 17:00 the day before? | Use |
|---|---|---|
| Schedule: date, weekday, carrier, flight number, route, scheduled dep/arr times, distance, scheduled elapsed time | Yes | Features |
| Scheduled departures per origin-hour / origin-day | Yes. BTS reports flights scheduled for the day, including later cancellations | Features |
| **`tail_number` and anything derived from it** | **No.** It's the aircraft actually flown, and it's missing exactly when the flight is cancelled | **Excluded** (v1 leak) |
| `month` | Yes, but it encodes one training year's weather | Excluded (lagged seasonal rates may come in Modify) |
| Yesterday's disruption rates by origin / destination / carrier, **from flights scheduled to land by 17:00** | Yes | Day-before features (`DECISION_CUTOFF_HHMM`) |
| Flight-day outcomes (dep/arr times and delays, taxi, cancelled, diverted, delay causes, actual/air time) | No | Target only |
| `daily_airport_stats.csv` | Only lagged and cut off at 17:00, which it isn't (it's full-day) | Excluded |
| `airlines.csv`, `airports.csv` departures/arrivals | **No.** They aggregate the whole period, test included | Excluded (`FORBIDDEN_FILES`) |

### F.5 Feasibility (two validation windows; test untouched)

In [6]:
from flightdelay.features import SCHEDULE_FEATURES, DAY_BEFORE_FEATURES
sched = assert_no_test_targets(d[d["period"].isin(["train", "valid"])]).drop_duplicates(
    ["flight_date", "airline_code", "flight_number", "origin", "sched_dep_time"])
# schedule counts and previous-day rates on the FULL schedule (all flights, incl. tail-less cancellations), then filter
feat = previous_day_rates(schedule_features(sched))
feat = feat[feat["tail_number"].notna() & feat["sched_elapsed_time"].gt(0)]   # scored population (assumption A-pop)
for col in ["carrier", "origin", "dest"]:
    feat[col] = feat[col].astype(str)

def per_day(dates, y, s, frac):
    """Per-day top-frac precision and lift (precision / that day's rate), one value per day."""
    prec, lift = [], []
    for _, idx in pd.Series(np.arange(len(y))).groupby(np.asarray(dates)):
        yi, si = y[idx.to_numpy()], s[idx.to_numpy()]
        k = max(1, int(round(frac * len(yi))))
        top = np.argsort(-si, kind="stable")[:k]
        if yi.mean() > 0:
            prec.append(yi[top].mean()); lift.append(yi[top].mean() / yi.mean())
    return np.array(prec), np.array(lift)

def block_bootstrap(x, block=7, n=2000, seed=config.SEED):
    """90% interval of the mean of a daily series, resampling 7-day blocks (daily rates are autocorrelated)."""
    rng = np.random.default_rng(seed)
    blocks = [x[i:i + block] for i in range(0, len(x), block)]
    means = [np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))]).mean() for _ in range(n)]
    return np.quantile(means, [0.05, 0.95])

def evaluate(train_rng, valid_rng, label):
    tr_all = feat[feat.flight_date.between(*train_rng)]
    va = feat[feat.flight_date.between(*valid_rng)].copy()
    tr = tr_all.sample(400_000, random_state=config.SEED).copy()
    for col in ["carrier", "origin", "dest"]:
        cats = pd.Index(sorted(tr[col].unique()))
        tr[col] = pd.Categorical(tr[col], categories=cats); va[col] = pd.Categorical(va[col].where(va[col].isin(cats)), categories=cats)
    y = va["disrupted"].to_numpy()
    hist = tr_all.groupby(["carrier", "dep_hour"], observed=True)["disrupted"].mean()
    sched_rule = np.array([hist.get(k, np.nan) for k in zip(va["carrier"].astype(str), va["dep_hour"])])
    scores = {"schedule rule (carrier × dep-hour history)": np.where(np.isnan(sched_rule), tr_all["disrupted"].mean(), sched_rule),
              "day-before rule (mean of yesterday's 3 rates)": va[DAY_BEFORE_FEATURES].mean(axis=1).fillna(tr_all["disrupted"].mean()).to_numpy()}
    for name, feats in [("LightGBM, schedule only", SCHEDULE_FEATURES), ("LightGBM, schedule + day-before", SCHEDULE_FEATURES + DAY_BEFORE_FEATURES)]:
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=100, subsample=0.8,
                               subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=config.SEED).fit(tr[feats], tr["disrupted"])
        scores[name] = m.predict_proba(va[feats])[:, 1]
    rows, daily = [], {}
    for name, s in scores.items():
        daily[name] = {f: per_day(va["flight_date"], y, s, f) for f in config.TOP_FRACTIONS}
        rows.append({"window": label, "scorer": name, "ROC-AUC": roc_auc_score(y, s), "PR-AUC": average_precision_score(y, s),
                     **{f"per-day top-{int(f * 100)}% precision": daily[name][f][0].mean() for f in config.TOP_FRACTIONS},
                     **{f"per-day top-{int(f * 100)}% lift": daily[name][f][1].mean() for f in config.TOP_FRACTIONS}})
    # model vs the BEST simple rule in this window, 7-day block bootstrap
    rules = ["schedule rule (carrier × dep-hour history)", "day-before rule (mean of yesterday's 3 rates)"]
    best = max(rules, key=lambda r: daily[r][0.02][1].mean())
    gap = daily["LightGBM, schedule + day-before"][0.02][1] - daily[best][0.02][1]
    lo, hi = block_bootstrap(gap)
    print(f"{label}: base rate {y.mean():.3f}, {len(gap)} days. Best simple rule: {best}. Model − best rule, per-day top-2% lift: "
          f"{gap.mean():+.2f} (90% 7-day-block interval {lo:+.2f} to {hi:+.2f}); model better on {np.mean(gap > 0):.0%} of days")
    return rows

res = pd.DataFrame(evaluate((config.TRAIN_START, config.TRAIN_END), (config.VALID_START, config.VALID_END), "main: 2025 → Jan–Apr 2026")
                   + evaluate(config.SUMMER_CHECK["train"], config.SUMMER_CHECK["valid"], "summer check: Jan–Apr 2025 → May–Aug 2025"))
display(res.set_index(["window", "scorer"]).round(3))

main: 2025 → Jan–Apr 2026: base rate 0.233, 120 days. Best simple rule: day-before rule (mean of yesterday's 3 rates). Model − best rule, per-day top-2% lift: +0.30 (90% 7-day-block interval +0.18 to +0.40); model better on 71% of days


summer check: Jan–Apr 2025 → May–Aug 2025: base rate 0.272, 123 days. Best simple rule: schedule rule (carrier × dep-hour history). Model − best rule, per-day top-2% lift: +0.19 (90% 7-day-block interval +0.10 to +0.28); model better on 67% of days


ROC-AUC  \
window                                    scorer                                                   
main: 2025 → Jan–Apr 2026                 schedule rule (carrier × dep-hour history)       0.610   
                                          day-before rule (mean of yesterday's 3 rates)    0.628   
                                          LightGBM, schedule only                          0.641   
                                          LightGBM, schedule + day-before                  0.685   
summer check: Jan–Apr 2025 → May–Aug 2025 schedule rule (carrier × dep-hour history)       0.661   
                                          day-before rule (mean of yesterday's 3 rates)    0.583   
                                          LightGBM, schedule only                          0.659   
                                          LightGBM, schedule + day-before                  0.675   

                                                                                         PR-AUC  \
window                                    scorer                                                  
main: 2025 → Jan–Apr 2026                 schedule rule (carrier × dep-hour history)      0.304   
                                          day-before rule (mean of yesterday's 3 rates)   0.359   
                                          LightGBM, schedule only                         0.345   
                                          LightGBM, schedule + day-before                 0.400   
summer check: Jan–Apr 2025 → May–Aug 2025 schedule rule (carrier × dep-hour history)      0.394   
                                          day-before rule (mean of yesterday's 3 rates)   0.330   
                                          LightGBM, schedule only                         0.398   
                                          LightGBM, schedule + day-before                 0.414   

                                                                                         per-day top-2% precision  \
window                                    scorer                                                                    
main: 2025 → Jan–Apr 2026                 schedule rule (carrier × dep-hour history)                        0.357   
                                          day-before rule (mean of yesterday's 3 rates)                     0.421   
                                          LightGBM, schedule only                                           0.447   
                                          LightGBM, schedule + day-before                                   0.488   
summer check: Jan–Apr 2025 → May–Aug 2025 schedule rule (carrier × dep-hour history)                        0.471   
                                          day-before rule (mean of yesterday's 3 rates)                     0.365   
                                          LightGBM, schedule only                                           0.499   
                                          LightGBM, schedule + day-before                                   0.516   

                                                                                         per-day top-10% precision  \
window                                    scorer                                                                     
main: 2025 → Jan–Apr 2026                 schedule rule (carrier × dep-hour history)                         0.342   
                                          day-before rule (mean of yesterday's 3 rates)                      0.368   
                                          LightGBM, schedule only                                            0.385   
                                          LightGBM, schedule + day-before                                    0.412   
summer check: Jan–Apr 2025 → May–Aug 2025 schedule rule (carrier × dep-hour history)                         0.447   
                                          day-before rule (mean of yesterday's 3 rates)             

**Feasibility verdict (honest numbers after removing the leak and applying the 17:00 cutoff):**

| Per-day top-2% lift (precision) | Main (→ Jan–Apr 2026) | Summer check (→ May–Aug 2025) |
|---|---|---|
| Schedule rule (carrier × dep-hour history) | 1.56 (0.36) | **1.74 (0.47)** |
| Day-before rule (yesterday's rates) | **1.89 (0.42)** | 1.39 (0.37) |
| LightGBM, schedule only | 1.98 (0.45) | 1.85 (0.50) |
| LightGBM, schedule + day-before | **2.19 (0.49)** | **1.93 (0.52)** |
| *Model − best simple rule (90% 7-day-block interval)* | *+0.30 (+0.18 to +0.40); better on 71% of days* | *+0.19 (+0.10 to +0.28); better on 67% of days* |

- **The best simple rule depends on the season.**
  - In winter–spring the day-before rule is strong. In summer it ranks flights within a day poorly (AUC 0.58), and the schedule pattern wins.
  - The daily disruption rate is somewhat less persistent in summer (day-to-day correlation 0.58 vs 0.69, per critic round 2). That's a *measured* difference, and the cause (e.g. local thunderstorms) is a hypothesis for Explore.
  - So the benchmark for every target is the **best of the two simple rules in that window**, not a fixed one.
- **The model beats both in both windows, but modestly:**
  - main window, vs the day-before rule: +0.30 per-day lift;
  - summer, vs the schedule rule: +0.19.
  - Both 7-day-block intervals exclude 0.
  - The v1 headline ("2.3× vs 1.5×") came from the leak, the uncut previous-day rates and an unfair baseline.
- **Implications for the SEMMA stages:**
  - *Sample* sets targets against the best simple rule.
  - *Modify* adds features that help in summer (lagged rates at hub level, the time-of-day bank structure, multi-day history).
  - *Assess* (a summer test) uses the summer-check window as its closest preview.


---
# Sample

- Learning curve: model quality vs sample size (50k → 2M), to justify the sample
- Stratified random sample (month × airline × target) with representativeness checks
- Sample saved to `data/sample/` (committed, so Colab works without the 314 MB file)

In [7]:
# TODO

---
# Explore

- Disruption by airline, hour, weekday, month, airport, route
- Delay-cause mix (insight only)
- Airport clustering by delay profile

In [8]:
# TODO

---
# Modify

- Lagged features (1–7 days, cut off at 17:00), scheduled congestion from the full schedule, holidays; a single reference clock (17:00 ET, time-zone-aware arrivals, 15-min margin; critic round 2 N3)
- Rare-category grouping, missing values, outliers
- Leakage audit re-run on the final feature list

In [9]:
# TODO

---
# Model

- Logistic regression, decision tree, neural network (MLP), gradient boosting
- Class-weighting, tuning on validation

In [10]:
# TODO

---
# Assess

- Single run on the sealed test months: ROC/PR, cumulative lift and gains (SAS style), calibration
- Breakdown by airline, airport, month; champion model

In [11]:
# TODO

---
# Score / deploy

- Saved pipeline + `score.py` batch scoring
- Small demo app: tomorrow's risk for a flight

In [12]:
# TODO